# Module 4 - Class 5: Regularization Study

**Dataset:** Telco Customer Churn  
**Objective:** Understand L1 (Lasso) vs L2 (Ridge) regularization through logistic regression.

### What you will learn
- What happens without regularization (overfitting risk)
- How L1 regularization zeros out irrelevant features
- How L2 regularization shrinks coefficients evenly
- Effect of the C parameter on model complexity

---

## 0. Setup

In [ ]:
# 0. Setup - Import all necessary libraries for the regularization study
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings

# Suppress unnecessary warnings to keep output clean
warnings.filterwarnings('ignore')

print("Setup complete.")


## 1. Load and Preprocess Data

In [ ]:
# 1. Load and Preprocess Data
# Load the Telco Customer Churn dataset from URL
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Fix TotalCharges column (it has some blank strings)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Standardize categorical columns with 'No internet service' etc.
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Convert target to binary (1 = Churn, 0 = No Churn)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Separate features and target
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

# Create preprocessing pipelines
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, cat_cols)
])

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Apply preprocessing
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Get feature names after one-hot encoding
ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
all_feature_names = numeric_cols + cat_feature_names

print(f"Train: {X_train_proc.shape}, Test: {X_test_proc.shape}")
print(f"Features: {len(all_feature_names)}")


## 2. No Regularization (Baseline)

In [ ]:
# 2. No Regularization (Baseline)
# Train Logistic Regression without any regularization (penalty=None)
# This serves as our baseline to compare overfitting risk
lr_none = LogisticRegression(penalty=None, max_iter=5000, random_state=42)
lr_none.fit(X_train_proc, y_train)
y_pred_none = lr_none.predict(X_test_proc)

print("No Regularization:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_none):.4f}")
print(f"  F1:       {f1_score(y_test, y_pred_none):.4f}")
print(f"  Non-zero coefficients: {np.sum(lr_none.coef_[0] != 0)} / {len(all_feature_names)}")


## 3. L1 Regularization (Lasso)

In [ ]:
# 3. L1 Regularization (Lasso)
# L1 adds absolute value penalty, which can drive some coefficients to exactly zero
lr_l1 = LogisticRegression(penalty='l1', solver='saga', C=1.0, max_iter=5000, random_state=42)
lr_l1.fit(X_train_proc, y_train)
y_pred_l1 = lr_l1.predict(X_test_proc)

print("L1 Regularization (Lasso):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_l1):.4f}")
print(f"  F1:       {f1_score(y_test, y_pred_l1):.4f}")
print(f"  Non-zero coefficients: {np.sum(lr_l1.coef_[0] != 0)} / {len(all_feature_names)}")

# Show which features were completely removed by L1
zeroed = [name for name, coef in zip(all_feature_names, lr_l1.coef_[0]) if coef == 0]
print(f"\nFeatures zeroed by L1: {zeroed if zeroed else 'None (try lower C)'}")


## 4. L2 Regularization (Ridge)

In [ ]:
# 4. L2 Regularization (Ridge)
# L2 adds squared penalty, which shrinks coefficients but doesn't set them to zero
lr_l2 = LogisticRegression(penalty='l2', C=1.0, max_iter=5000, random_state=42)
lr_l2.fit(X_train_proc, y_train)
y_pred_l2 = lr_l2.predict(X_test_proc)

print("L2 Regularization (Ridge):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_l2):.4f}")
print(f"  F1:       {f1_score(y_test, y_pred_l2):.4f}")
print(f"  Non-zero coefficients: {np.sum(lr_l2.coef_[0] != 0)} / {len(all_feature_names)}")


## 5. Coefficient Comparison Heatmap

In [ ]:
# 5. Coefficient Comparison Heatmap
# Compare coefficients from all three models side by side
coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'No Reg': lr_none.coef_[0],
    'L1 (Lasso)': lr_l1.coef_[0],
    'L2 (Ridge)': lr_l2.coef_[0]
}).set_index('Feature')

# Sort by importance in the unregularized model
coef_df = coef_df.loc[coef_df['No Reg'].abs().sort_values(ascending=False).index]

plt.figure(figsize=(10, max(8, len(all_feature_names) * 0.3)))
sns.heatmap(
    coef_df,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    linewidths=0.5
)
plt.title('Coefficient Comparison: No Reg vs L1 vs L2', fontsize=13)
plt.tight_layout()
plt.show()

print("\nKey observation: L1 sets some coefficients to exactly 0 (feature selection).")
print("L2 shrinks all coefficients toward 0 but never exactly to 0.")


## 6. Effect of C Parameter

C = inverse regularization strength.  
- **Small C** = strong regularization (simpler model)  
- **Large C** = weak regularization (closer to no regularization)

In [ ]:
# 6. Effect of C Parameter
# C is the inverse of regularization strength. Smaller C = stronger regularization
C_values = [0.01, 0.1, 1, 10, 100]
results_l1 = []
results_l2 = []

for c in C_values:
    # L1 with different C
    model_l1 = LogisticRegression(penalty='l1', solver='saga', C=c, max_iter=5000, random_state=42)
    model_l1.fit(X_train_proc, y_train)
    results_l1.append({
        'C': c,
        'Accuracy': accuracy_score(y_test, model_l1.predict(X_test_proc)),
        'F1': f1_score(y_test, model_l1.predict(X_test_proc)),
        'Non-zero Coefs': np.sum(model_l1.coef_[0] != 0)
    })

    # L2 with different C
    model_l2 = LogisticRegression(penalty='l2', C=c, max_iter=5000, random_state=42)
    model_l2.fit(X_train_proc, y_train)
    results_l2.append({
        'C': c,
        'Accuracy': accuracy_score(y_test, model_l2.predict(X_test_proc)),
        'F1': f1_score(y_test, model_l2.predict(X_test_proc))
    })

df_l1 = pd.DataFrame(results_l1)
df_l2 = pd.DataFrame(results_l2)

print("L1 Results:")
print(df_l1.to_string(index=False))
print("\nL2 Results:")
print(df_l2.to_string(index=False))


In [ ]:
# Plot the effect of C on model performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(C_values, df_l1['Accuracy'], 'o-', label='L1 (Lasso)', color='#e74c3c', linewidth=2)
axes[0].plot(C_values, df_l2['Accuracy'], 's-', label='L2 (Ridge)', color='#3498db', linewidth=2)
axes[0].set_xscale('log')
axes[0].set_xlabel('C (inverse regularization strength)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs C')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 plot
axes[1].plot(C_values, df_l1['F1'], 'o-', label='L1 (Lasso)', color='#e74c3c', linewidth=2)
axes[1].plot(C_values, df_l2['F1'], 's-', label='L2 (Ridge)', color='#3498db', linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('C (inverse regularization strength)')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score vs C')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of Regularization Strength', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# L1 feature selection effect
plt.figure(figsize=(8, 4))
plt.plot(C_values, df_l1['Non-zero Coefs'], 'o-', color='#e74c3c', linewidth=2, markersize=8)
plt.xscale('log')
plt.xlabel('C value')
plt.ylabel('Number of Non-zero Coefficients')
plt.title('L1: Feature Selection Effect (more features as C increases)')
plt.axhline(y=len(all_feature_names), color='gray', linestyle='--', label=f'Total features ({len(all_feature_names)})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. TODO: When to Use L1 vs L2?

Based on the results and plots above, answer these questions:

1. When would you use L1 (Lasso) regularization? Give a concrete scenario.
2. When would you use L2 (Ridge) regularization? Give a concrete scenario.
3. What happens when C is very small (e.g., 0.01)? What about very large (e.g., 100)?
4. How does L1 act as automatic feature selection?

**TODO Answer:**

1. **L1 (Lasso) is best used** when we have a large number of features and we suspect that only a subset of them are truly important. For example, in a marketing dataset with hundreds of customer features, L1 can automatically select the most relevant ones by setting irrelevant coefficients to zero.

2. **L2 (Ridge) is preferred** when all features are likely to contribute to the prediction and we want to prevent any single feature from having too much influence. It's commonly used in cases where features are correlated, like in medical or financial datasets.

3. When C is very small (e.g., 0.01), strong regularization is applied. This results in simpler models with lower accuracy but better generalization. When C is very large (e.g., 100), regularization is weak, and the model behaves almost like no regularization, which can lead to overfitting.

4. L1 acts as automatic feature selection because the L1 penalty (absolute value of coefficients) can drive some coefficients exactly to zero, effectively removing those features from the model.

---
## Summary

| Concept | Details |
|---------|--------|
| No Regularization | All coefficients free — risk of overfitting on noisy features |
| L1 (Lasso) | Adds \|w\| penalty — drives some coefficients to exactly 0 (feature selection) |
| L2 (Ridge) | Adds w^2 penalty — shrinks all coefficients but none reach 0 |
| C parameter | Inverse regularization strength: small C = strong penalty, large C = weak penalty |
| When L1 | Many features, suspect most are irrelevant — want automatic feature selection |
| When L2 | All features likely contribute — want to prevent any single feature from dominating |